In [1]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='MedHouseVal')

# YOUR JOB: 
# - Print X.shape and y.shape — confirm 20640 rows, 8 features
# - Run X.head() in a cell to see the first 5 rows
# - Run X.describe() to see summary statistics for each feature
# - Run y.describe() to see the target distribution

In [2]:
X.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [3]:
print(X.shape)       # should be (20640, 8)
print(y.shape)       # should be (20640,)
X.describe()         # summary stats per feature

(20640, 8)
(20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


In [4]:
y.describe() 

count    20640.000000
mean         2.068558
std          1.153956
min          0.149990
25%          1.196000
50%          1.797000
75%          2.647250
max          5.000010
Name: MedHouseVal, dtype: float64

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"train: {X_train.shape}, test: {X_test.shape}")

train: (16512, 8), test: (4128, 8)


In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Three trees with very different complexity
trees = {
    "shallow (max_depth=3)": DecisionTreeRegressor(max_depth=3, random_state=42),
    "medium (max_depth=8)": DecisionTreeRegressor(max_depth=8, random_state=42),
    "giant (no limit)":     DecisionTreeRegressor(random_state=42),
}

print(f"{'tree':<25} | {'leaves':>7} | {'train MSE':>10} | {'test MSE':>10}")
print("-" * 65)
for name, tree in trees.items():
    tree.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, tree.predict(X_train))
    test_mse  = mean_squared_error(y_test,  tree.predict(X_test))
    print(f"{name:<25} | {tree.get_n_leaves():>7} | {train_mse:>10.4f} | {test_mse:>10.4f}")

tree                      |  leaves |  train MSE |   test MSE
-----------------------------------------------------------------
shallow (max_depth=3)     |       8 |     0.6177 |     0.6424
medium (max_depth=8)      |     243 |     0.3206 |     0.4220
giant (no limit)          |   15854 |     0.0000 |     0.4952


In [7]:
import numpy as np

# Get the pruning path from the giant unpruned tree
giant = trees["giant (no limit)"]   # already trained
path = giant.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas

print(f"sklearn proposed {len(alphas)} alpha values to try")
print(f"this will train {len(alphas)} trees — may take 30-60 seconds")

# For efficiency, sample every Nth alpha (full path is overkill for our purposes)
step = max(1, len(alphas) // 30)
sampled_alphas = alphas[::step]
print(f"sampling {len(sampled_alphas)} of them (every {step}th)")

sklearn proposed 14833 alpha values to try
this will train 14833 trees — may take 30-60 seconds
sampling 31 of them (every 494th)


In [8]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Train one tree per sampled alpha, record train and test MSE
results = []
for i, a in enumerate(sampled_alphas):
    t = DecisionTreeRegressor(ccp_alpha=a, random_state=42)
    t.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, t.predict(X_train))
    test_mse  = mean_squared_error(y_test,  t.predict(X_test))
    results.append({
        "alpha": a,
        "leaves": t.get_n_leaves(),
        "train_mse": train_mse,
        "test_mse": test_mse,
    })
    print(f"  [{i+1}/{len(sampled_alphas)}] alpha={a:.5f}, leaves={t.get_n_leaves()}, test MSE={test_mse:.4f}")

# Find the winner
best = min(results, key=lambda r: r["test_mse"])
print(f"\n*** BEST: alpha={best['alpha']:.5f}, leaves={best['leaves']}, test MSE={best['test_mse']:.4f} ***")
print(f"    (medium tree from step 3 had test MSE = 0.4220)")

  [1/31] alpha=0.00000, leaves=15854, test MSE=0.4952
  [2/31] alpha=0.00000, leaves=15339, test MSE=0.4952
  [3/31] alpha=0.00000, leaves=14850, test MSE=0.4952
  [4/31] alpha=0.00000, leaves=14352, test MSE=0.4952
  [5/31] alpha=0.00000, leaves=13859, test MSE=0.4952
  [6/31] alpha=0.00000, leaves=13364, test MSE=0.4953
  [7/31] alpha=0.00000, leaves=12868, test MSE=0.4953
  [8/31] alpha=0.00000, leaves=12374, test MSE=0.4954
  [9/31] alpha=0.00000, leaves=11878, test MSE=0.4953
  [10/31] alpha=0.00000, leaves=11380, test MSE=0.4953
  [11/31] alpha=0.00000, leaves=10880, test MSE=0.4952
  [12/31] alpha=0.00000, leaves=10381, test MSE=0.4952
  [13/31] alpha=0.00000, leaves=9884, test MSE=0.4953
  [14/31] alpha=0.00000, leaves=9384, test MSE=0.4951
  [15/31] alpha=0.00000, leaves=8878, test MSE=0.4952
  [16/31] alpha=0.00000, leaves=8370, test MSE=0.4944
  [17/31] alpha=0.00000, leaves=7852, test MSE=0.4935
  [18/31] alpha=0.00000, leaves=7343, test MSE=0.4932
  [19/31] alpha=0.00000, 

In [9]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import pandas as pd

# Retrain the winning tree so we can inspect it
best_alpha = 0.00015  # use the alpha that won
best_tree = DecisionTreeRegressor(ccp_alpha=best_alpha, random_state=42)
best_tree.fit(X_train, y_train)

importances = pd.Series(
    best_tree.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Feature importances (higher = more useful for prediction):\n")
print(importances)

Feature importances (higher = more useful for prediction):

MedInc        0.580630
AveOccup      0.130708
Latitude      0.089187
Longitude     0.074315
HouseAge      0.045783
AveRooms      0.044390
Population    0.017781
AveBedrms     0.017206
dtype: float64


The medium-depth tree (max_depth=8, 243 leaves) outperformed both the shallow tree (underfit, MSE 0.6424) and the giant unpruned tree (overfit, train MSE 0.0 but test MSE 0.4952), landing at test MSE 0.4220. Cost-complexity pruning then improved on this further: with alpha = 0.00015, the pruned tree had 679 leaves and test MSE 0.4085 — about a 3% improvement over the manually-tuned medium tree, but achieved automatically without guessing at a max_depth value. The most important feature by a wide margin was median income (58%), followed by average occupancy (13%) and location via latitude/longitude (16% combined). Bedrooms ranked dead last (1.7%), which is counterintuitive at first but makes sense: across districts, what varies most is neighborhood type, not house size.